# 02 Cechy – klasyfikacja emocji i sentymentu

## 1. Konfiguracja

In [4]:
import sys
sys.path.append("..")

import time

import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import pipeline

from src.config import EMOTION_MODEL, EMOTIONS, PROCESSED_DIR, SENTIMENT_MODEL

SENTIMENT_LABELS = ["negative", "neutral", "positive"]
BATCH_SIZE = 32
EMOTIONS_DIR = PROCESSED_DIR / "emotions"
SENTIMENT_DIR = PROCESSED_DIR / "sentiment"
for folder in (EMOTIONS_DIR, SENTIMENT_DIR):
    folder.mkdir(parents=True, exist_ok=True)
device = 0 if torch.cuda.is_available() else -1
print("urządzenie:", "GPU" if device == 0 else "CPU")

urządzenie: GPU


## 2. Wczytanie próby i modeli

In [5]:
sample = pd.read_parquet(PROCESSED_DIR / "sample.parquet", columns=["id", "subreddit", "cycle", "text"])
emotion_pipeline = pipeline("text-classification", model=EMOTION_MODEL, top_k=None, device=device)
sentiment_pipeline = pipeline("text-classification", model=SENTIMENT_MODEL, top_k=None, device=device)
sample.shape

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 23320.38it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(794519, 4)

## 3. Klasyfikacja – osobna komórka na subreddit

In [6]:
def classify_subreddit(subreddit, cycles):
    """Emocje i sentyment dla podanych cykli jednego subredditu, zapis po każdym cyklu."""
    rows = sample[(sample["subreddit"] == subreddit) & sample["cycle"].isin(cycles)]
    total_chars = rows["text"].str.len().sum()
    done_chars = 0
    start_time = time.time()
    for cycle in cycles:
        group = rows[rows["cycle"] == cycle]
        group = group.iloc[group["text"].str.len().argsort().values]
        texts = group["text"].tolist()
        print(f"{subreddit} {cycle}: {len(texts)} tekstów")
        for name, model_pipeline, folder, labels in (("emocje", emotion_pipeline, EMOTIONS_DIR, EMOTIONS),
                                                     ("sentyment", sentiment_pipeline, SENTIMENT_DIR, SENTIMENT_LABELS)):
            stream = model_pipeline((text for text in texts), batch_size=BATCH_SIZE, truncation=True, max_length=512)
            outputs = list(tqdm(stream, total=len(texts), desc=name))
            probabilities = pd.DataFrame([{item["label"].lower(): item["score"] for item in output} for output in outputs])[labels]
            probabilities.insert(0, "id", group["id"].values)
            probabilities.to_parquet(folder / f"{subreddit}_{cycle}_{folder.name}.parquet", index=False)
        torch.cuda.empty_cache()  # oddaje zapas pamięci GPU po grupie, żeby przez noc nie narastał
        done_chars += group["text"].str.len().sum()
        elapsed = time.time() - start_time
        left = elapsed / done_chars * (total_chars - done_chars)
        print(f"    minęło {elapsed / 3600:.1f} h, zostało ok. {left / 3600:.1f} h")

### 3.1 r/democrats

In [7]:
classify_subreddit("democrats", [2012, 2016, 2020, 2024])

democrats 2012: 11519 tekstów


sentyment: 100%|██████████| 11519/11519 [02:57<00:00, 64.79it/s]


    minęło 0.1 h, zostało ok. 0.4 h
democrats 2016: 30285 tekstów


sentyment: 100%|██████████| 30285/30285 [08:17<00:00, 60.90it/s] 


    minęło 0.3 h, zostało ok. 0.3 h
democrats 2020: 31500 tekstów


sentyment: 100%|██████████| 31500/31500 [05:53<00:00, 89.03it/s] 


    minęło 0.4 h, zostało ok. 0.1 h
democrats 2024: 31200 tekstów


sentyment: 100%|██████████| 31200/31200 [05:28<00:00, 94.92it/s] 


    minęło 0.6 h, zostało ok. 0.0 h


### 3.2 r/Liberal

In [8]:
classify_subreddit("Liberal", [2012, 2016, 2020, 2024])

Liberal 2012: 16851 tekstów


sentyment: 100%|██████████| 16851/16851 [06:19<00:00, 44.44it/s] 


    minęło 0.2 h, zostało ok. 0.7 h
Liberal 2016: 30478 tekstów


sentyment: 100%|██████████| 30478/30478 [08:50<00:00, 57.48it/s] 


    minęło 0.4 h, zostało ok. 0.4 h
Liberal 2020: 29924 tekstów


sentyment: 100%|██████████| 29924/29924 [05:41<00:00, 87.58it/s] 


    minęło 0.5 h, zostało ok. 0.2 h
Liberal 2024: 30867 tekstów


sentyment: 100%|██████████| 30867/30867 [05:58<00:00, 86.05it/s] 


    minęło 0.7 h, zostało ok. 0.0 h


### 3.3 r/Republican

In [9]:
classify_subreddit("Republican", [2012, 2016, 2020, 2024])

Republican 2012: 24002 tekstów


sentyment: 100%|██████████| 24002/24002 [11:07<00:00, 35.95it/s] 


    minęło 0.3 h, zostało ok. 0.8 h
Republican 2016: 30891 tekstów


sentyment: 100%|██████████| 30891/30891 [07:54<00:00, 65.11it/s] 
[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


    minęło 0.5 h, zostało ok. 0.4 h
Republican 2020: 31128 tekstów


sentyment: 100%|██████████| 31128/31128 [07:09<00:00, 72.42it/s] 


    minęło 0.7 h, zostało ok. 0.2 h
Republican 2024: 31200 tekstów


sentyment: 100%|██████████| 31200/31200 [06:36<00:00, 78.74it/s] 


    minęło 0.8 h, zostało ok. 0.0 h


### 3.4 r/Conservative

In [10]:
classify_subreddit("Conservative", [2012, 2016, 2020, 2024])

Conservative 2012: 31000 tekstów


sentyment: 100%|██████████| 31000/31000 [09:26<00:00, 54.74it/s] 


    minęło 0.2 h, zostało ok. 0.5 h
Conservative 2016: 31500 tekstów


sentyment: 100%|██████████| 31500/31500 [08:07<00:00, 64.61it/s] 


    minęło 0.4 h, zostało ok. 0.3 h
Conservative 2020: 31500 tekstów


sentyment: 100%|██████████| 31500/31500 [07:41<00:00, 68.19it/s] 


    minęło 0.6 h, zostało ok. 0.2 h
Conservative 2024: 31200 tekstów


sentyment: 100%|██████████| 31200/31200 [05:48<00:00, 89.62it/s] 


    minęło 0.8 h, zostało ok. 0.0 h


### 3.5 r/running

In [11]:
classify_subreddit("running", [2012, 2014, 2016, 2018, 2020, 2022, 2024])

running 2012: 31200 tekstów


sentyment: 100%|██████████| 31200/31200 [06:47<00:00, 76.62it/s] 


    minęło 0.2 h, zostało ok. 0.8 h
running 2014: 15900 tekstów


sentyment: 100%|██████████| 15900/15900 [03:53<00:00, 68.08it/s]


    minęło 0.3 h, zostało ok. 0.8 h
running 2016: 31500 tekstów


sentyment: 100%|██████████| 31500/31500 [07:43<00:00, 67.98it/s] 


... (5 outputs left out)

sentyment: 100%|██████████| 15900/15900 [05:04<00:00, 52.17it/s]


    minęło 1.0 h, zostało ok. 0.2 h
running 2024: 31200 tekstów


sentyment: 100%|██████████| 31200/31200 [11:30<00:00, 45.16it/s] 


    minęło 1.3 h, zostało ok. 0.0 h


### 3.6 r/languagelearning

In [12]:
classify_subreddit("languagelearning", [2012, 2014, 2016, 2018, 2020, 2022, 2024])

languagelearning 2012: 24474 tekstów


sentyment: 100%|██████████| 24474/24474 [06:55<00:00, 58.92it/s] 


    minęło 0.2 h, zostało ok. 1.0 h
languagelearning 2014: 15900 tekstów


sentyment: 100%|██████████| 15900/15900 [04:25<00:00, 59.79it/s]


    minęło 0.3 h, zostało ok. 0.9 h
languagelearning 2016: 31500 tekstów


sentyment: 100%|██████████| 31500/31500 [09:50<00:00, 53.35it/s] 


... (5 outputs left out)

sentyment: 100%|██████████| 15900/15900 [04:44<00:00, 55.85it/s]


    minęło 1.0 h, zostało ok. 0.2 h
languagelearning 2024: 31200 tekstów


sentyment: 100%|██████████| 31200/31200 [09:08<00:00, 56.83it/s] 


    minęło 1.2 h, zostało ok. 0.0 h


## 4. Kontrola

In [13]:
emotions = pd.concat(pd.read_parquet(path) for path in sorted(EMOTIONS_DIR.glob("*.parquet")))
sentiment = pd.concat(pd.read_parquet(path) for path in sorted(SENTIMENT_DIR.glob("*.parquet")))
print("pliki:", len(list(EMOTIONS_DIR.glob("*.parquet"))), len(list(SENTIMENT_DIR.glob("*.parquet"))))
print("wiersze:", len(emotions), len(sentiment), len(sample))
print("id unikalne:", emotions["id"].is_unique, sentiment["id"].is_unique)
print("te same id co w próbie:", set(emotions["id"]) == set(sample["id"]), set(sentiment["id"]) == set(sample["id"]))
print("kolumny:", list(emotions.columns), list(sentiment.columns))
pd.DataFrame({"emocje": emotions[EMOTIONS].sum(axis=1).describe(),
              "sentyment": sentiment[SENTIMENT_LABELS].sum(axis=1).describe()})

pliki: 30 30
wiersze: 794519 794519 794519
id unikalne: True True
te same id co w próbie: True True
kolumny: ['id', 'anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise'] ['id', 'negative', 'neutral', 'positive']


,emocje,sentyment
count,7.945190e+05,7.945190e+05
mean,1.000000e+00,1.000000e+00
std,5.404950e-08,4.030876e-08
min,9.999997e-01,9.999999e-01
25%,1.000000e+00,1.000000e+00
50%,1.000000e+00,1.000000e+00
75%,1.000000e+00,1.000000e+00
max,1.000000e+00,1.000000e+00
